In [ ]:
from bs4 import BeautifulSoup

from Data_Analysis import *
from Wanted_Crawling import *

ModuleNotFoundError: No module named 'Crawling'

In [ ]:
import requests
df_all = pd.DataFrame()

# 1. 가져올 URL 설정
base_url = "https://www.wanted.co.kr/company/"
for company_id in tqdm(range(1, 53300)):  # 예시로 1부터 99까지의 회사 ID를 사용
    company_url = f"{base_url}{company_id}"
    # 2. GET 요청 보내기 (타임아웃을 지정하는 것이 안전)
    try:
        response = requests.get(company_url, timeout=10)
        response.raise_for_status()  # 상태 코드가 200번대가 아니면 예외 발생
    except requests.exceptions.RequestException as e:
        print(f"요청 중 오류 발생: {e}")
    else:
        # 3. response.text로 HTML 전체 문자열을 얻을 수 있음
        html = response.text

    soup = BeautifulSoup(html, "html.parser")

    # ────────────────────────────────────────────────────────────────────────────────
    # 1) HTML 문자열을 BeautifulSoup으로 파싱
    #    (실제 사용 시 requests.get(url).text 등을 html_str에 할당해 주면 됩니다.)


    soup = BeautifulSoup(html, "html.parser")


    company_desc = soup.find("div", class_="wds-1q5hgpy")
    if company_desc:
        company_desc = company_desc.get_text(strip=True).replace("\n", "").replace("\t", "")
    else:
        company_desc = None

    # ────────────────────────────────────────────────────────────────────────────────
    # 2) “회사 정보” 섹션(data-testid="company-info")을 기준으로
    info_section = soup.find("div", {"data-testid": "company-info"})
    if info_section:
        # 2-1) 회사명
        h1_tag = info_section.find("h1")
        company_name = h1_tag.get_text(strip=True) if h1_tag else None

        # 2-2) 업종(산업)·지역
        info_div = info_section.find("div", class_="wds-12w9rv9")
        if info_div:
            spans = info_div.find_all("span", class_="wds-ilos43")
            industry = spans[0].get_text(strip=True) if len(spans) > 0 else None
            location = spans[1].get_text(strip=True) if len(spans) > 1 else None

            # 2-3) 업력과 설립년도
            age_tag = info_div.find("span", class_="wds-bhakrp")
            company_age = age_tag.get_text(strip=True) if age_tag else None

            founded_tag = info_div.find("span", class_="wds-ugwfgc")
            founded_year = founded_tag.get_text(strip=True) if founded_tag else None
    else:
        company_name = industry = location = company_age = founded_year = None

    # ────────────────────────────────────────────────────────────────────────────────
    # 3) “기업 정보” 섹션( <h2>기업 정보</h2> )을 찾아 그 안의 <dt> / <dd> 쌍 추출
    average_salary = employment_insured = national_pension = total_sales = None

    company_info_header = soup.find("h2", string="기업 정보")
    if company_info_header:
        info_table = company_info_header.find_next_sibling("div", class_="CompanyInfoTable_wrapper__xI_Gq")
        if info_table:
            dt_tags = info_table.find_all("dt", class_="CompanyInfoTable_definition__dt__hMyz7")
            dd_tags = info_table.find_all("dd", class_="CompanyInfoTable_definition__dd__oV9wp")
            info_dict = {}
            for dt, dd in zip(dt_tags, dd_tags):
                key = dt.get_text(strip=True)
                value = dd.get_text(strip=True).replace("\n", "").replace("\t", "")
                info_dict[key] = value

            average_salary = info_dict.get("평균연봉")
            employment_insured = info_dict.get("고용보험 가입 사원수")
            national_pension = info_dict.get("국민연금 가입 사원수")
            total_sales = info_dict.get("매출액")

    # ────────────────────────────────────────────────────────────────────────────────
    detailed_address = None
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except (json.JSONDecodeError, TypeError):
            continue
        if data.get("@type") == "Organization":
            detailed_address = data.get("address", {}).get("name")
            break


    df = pd.DataFrame({
        "회사명": [company_name],
        "회사 소개": [company_desc],
        "업종(산업)": [industry],
        "지역": [location],
        "업력": [company_age],
        "설립년도": [founded_year],
        "평균연봉": [average_salary],
        "고용보험 가입 사원수": [employment_insured],
        "국민연금 가입 사원수": [national_pension],
        "매출액": [total_sales],
        "상세 위치": [detailed_address]
    })

    df_all = pd.concat([df_all, df], ignore_index=True)



In [ ]:
df_all.fillna('정보 없음', inplace=True) # 결측값을 '없음'으로 대체)
df_all

In [ ]:
df_improve = toImproveDataFrame(df_all)
df_improve

In [ ]:
split_columns = ['회사 소개']

for column in split_columns:
    df_improve[column] = df_improve[column].apply(
        lambda x: x.split('\n') if isinstance(x, str) else x)

In [ ]:
df_improve

In [ ]:
df_improve = toImproveDataFrame(df_improve)
df_improve